In [1]:
# import the necessery package
import numpy as np
import os
import argparse
import cv2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model

In [2]:
# construct the argument parser and parse the arguments

ap = argparse.ArgumentParser()
ap.add_argument("-i", "--image", required = True, help = "path to input image")
ap.add_argument("-f", "--face", type = str, default = "face_detector", help = "path to face detector model directory")
ap.add_argument("-m", "-model", type = str, default = "mask_detector.model", help = "path to trained face mask detector model")
ap.add_argument("-c", "--confidence", type = float, default = 0.5, help = "minimum probability to filter weak detection")
args = vars(ap.parse_args())

usage: ipykernel_launcher.py [-h] -i IMAGE [-f FACE] [-m M] [-c CONFIDENCE]
ipykernel_launcher.py: error: the following arguments are required: -i/--image


SystemExit: 2

C:\Anaconda3\envs\aiml\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
# load the serialized face detector model from disk

print("[INFO] loading face detector model...")
prototxtPath = os.path.join("face_detector", "deploy.prototxt")
weightsPath = os.path.join("face_detector", "res10_300x300_ssd_iter_140000.caffemodel")
net = cv2.dnn.readNet(prototxtPath, weightsPath)



[INFO] loading face detector model...


In [4]:
print(prototxtPath)

face_detector\deploy.prototxt


In [5]:
print(weightsPath)

face_detector\res10_300x300_ssd_iter_140000.caffemodel


In [6]:
print(net)

< cv2.dnn.Net 000001709E6DEA50>


In [7]:
# load the face mask detector model from disk

print("[INFO] loading face mask detector model...")
model = load_model("model_mask_detector.h5")

[INFO] loading face mask detector model...


In [11]:
# load the input from the disk, clone it and grab the image spatial dimensions

image = cv2.imread("dataset/with_mask/0_0_12.jpg")
orig = image.copy()
(h, w) = image.shape[:2]

In [18]:
print(h, w)

133 95


In [15]:
# construct the blob from the image:
blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))

In [17]:
print(blob.shape)

(1, 3, 300, 300)


In [19]:
# pass the blob through the network and obtain the face detection:

print("[INFO] computing face detections...")

net.setInput(blob)
detections = net.forward()

[INFO] computing face detections...


In [20]:
print(net)

< cv2.dnn.Net 000001709E6DEA50>


In [22]:
print(detections.shape)

(1, 1, 200, 7)


In [36]:
# loop over the detections:

CONFIDENCE = 0.5
for i in range(0, detections.shape[2]):
    # extract the confidence associated with the detection
    confidence = detections[0, 0, i, 2]

    #filter out weak detection by ensuring the confidence is greater then the minimum confidence: 
    if confidence > CONFIDENCE:
        # compute the (X,Y)-coordinate of the bounding box for the object:
        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
        (startX, startY, endX, endY) = box.astype("int")

        # ensure the bounding boxes fall within the dimentions of the frame:
        (startX, startY) = (max(0, startX), max(0, startY))
        (endX, endY) = (min(w - 1, endX), min(h - 1, endY))
		

        # extract the face ROI, convert it from BGR to RGB channel ordering, resize it to (224x224), and preprocess it
        face =image[startY:endY, startX:endY]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, (224,224))
        face = img_to_array(face)
        face = preprocess_input(face)
        face = np.expand_dims(face, axix=0)
        
        # pass the face through the model to determine if the face has a mask or not:
        (mask, withoutMask) = model.predict(face)[0]

        # determine the class label and coloer will use to draw the bounding box and text 
        label = "Mask" if mask > withoutMask else "No Mask"
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)

        # include the probability in the label:
        label = "{}: {:.2f}%".format(label, max(mask, withoutMask) * 100)

        # display the label and bounding box rectangle on the output frame :
        cv2.putText(image, label, (startX, startY - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
        cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)
        
        

In [35]:
# show the output image
cv2.imshow("Output", image)
cv2.waitKey(0)

-1